In [30]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Loop Unrolling

In [31]:
render_code("swap.c")

// swap.c:1-65 (65 lines)
#include <stdint.h>
#include <stdio.h>
#include <sys/time.h>
#include <stdlib.h>
void (*swap_array)(uint64_t* a, uint64_t* b, uint64_t size);

void inline swap(uint64_t* a, uint64_t* b)
{
    int temp = *a;
    *a = *b;
    *b = temp;
} 

void swap_array_1(uint64_t *a, uint64_t *b, uint64_t size)
{
    for (uint64_t i = 0; i < size; ++i) {
        swap(&a[i],&b[i]);
    }

}
void swap_array_2(uint64_t *a, uint64_t *b, uint64_t size)
{
    for (uint64_t i = 0; i < size; i++) {
        swap(&a[i],&b[i]);
        i++;
        swap(&a[i],&b[i]);
    }
}

void __attribute__((optimize("unroll-loops")))swap_array_3(uint64_t *a, uint64_t *b, uint64_t size)
{
    uint64_t real_size = size / 4;
    for (uint64_t i = 0; i < real_size*4; ++i) {
        swap(&a[i],&b[i]);
    }
}




int main(int argc, char **argv)
{
    unsigned array_size = 131072;
    uint64_t *data_a, *data_b;
    struct timeval time_start, time_end;
    array_size = (unsigned)atoi(argv[1]);
    if(argc > 2 && atoi(argv[2])==1)
        swap_array = swap_array_2;
    else if(argc > 2 && atoi(argv[2])==2)
        swap_array = swap_array_3;
    else
        swap_array = swap_array_1;
    data_a = (uint64_t *)malloc(sizeof(uint64_t)*array_size);
    data_b = (uint64_t *)malloc(sizeof(uint64_t)*array_size);
    for (unsigned i = 0; i < array_size; ++i)
        data_a[i] = rand();
    for (unsigned i = 0; i < array_size; ++i)
        data_b[i] = rand();
   gettimeofday(&time_start, NULL);
    swap_array(data_a,data_b,array_size);
   gettimeofday(&time_end, NULL);
   fprintf(stderr, "data_a[array_size/2] = %lu\t", data_a[rand()%131072]);
   fprintf(stderr, "swapped %lf seconds\n",((time_end.tv_sec * 1000000 + time_end.tv_usec) - (time_start.tv_sec * 1000000 + time_start.tv_usec))/1000000.0);
   return 0;
}

In [29]:
! gcc -S -O3 -mno-avx swap.c
compare([do_render_code("swap.s",show=["swap_array_1:",".L1:"]),do_render_code("swap.s", show=["swap_array_2:","L20:"]),do_render_code("swap.s", show=["swap_array_3:","ret"])])

In [37]:
! gcc -O3 -mno-avx swap.c -o swap
! echo "Without unrolling"; for i in {1..10} ; do ./swap 8388608 0; done;
! echo "Manual unrolling"; for i in {1..10} ; do ./swap 8388608 1; done;
! echo "Compiler unrolling"; for i in {1..10} ; do ./swap 8388608 2; done;

Without unrolling
data_a[array_size/2] = 1371396557	swapped 0.006316 seconds
data_a[array_size/2] = 1371396557	swapped 0.006670 seconds
data_a[array_size/2] = 1371396557	swapped 0.006206 seconds
data_a[array_size/2] = 1371396557	swapped 0.006709 seconds
data_a[array_size/2] = 1371396557	swapped 0.006350 seconds
data_a[array_size/2] = 1371396557	swapped 0.006372 seconds
data_a[array_size/2] = 1371396557	swapped 0.006363 seconds
data_a[array_size/2] = 1371396557	swapped 0.006757 seconds
data_a[array_size/2] = 1371396557	swapped 0.006774 seconds
data_a[array_size/2] = 1371396557	swapped 0.006751 seconds
Manual unrolling
data_a[array_size/2] = 1371396557	swapped 0.006769 seconds
data_a[array_size/2] = 1371396557	swapped 0.006200 seconds
data_a[array_size/2] = 1371396557	swapped 0.006624 seconds
data_a[array_size/2] = 1371396557	swapped 0.006733 seconds
data_a[array_size/2] = 1371396557	swapped 0.006629 seconds
data_a[array_size/2] = 1371396557	swapped 0.006601 seconds
data_a[array_size/2] 